# High-Level Backtesting of the ICT Strategy

In [1]:
# imports
import time
import pandas as pd
from nautilus_trader.backtest.config import BacktestVenueConfig, BacktestDataConfig, BacktestRunConfig
from nautilus_trader.backtest.engine import BacktestResult, BacktestEngine, BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.common.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model import BarType, Bar, Venue, InstrumentId
from nautilus_trader.model.enums import OmsType
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.persistence.config import DataCatalogConfig
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading import trader
from nautilus_trader.trading.config import ImportableStrategyConfig
import sys
from pathlib import Path

In [2]:
sys.path.append(str(Path.cwd().parent))

catalog = ParquetDataCatalog("../catalog")

start_ns = dt_to_unix_nanos(pd.Timestamp("2025-07-09"))
end_ns = dt_to_unix_nanos(pd.Timestamp("2025-10-22"))

instrument = TestInstrumentProvider.es_future(2025, 12)
instrument_id = instrument.id.value

# Configure backtesting
venue = BacktestVenueConfig(
    name="GLBX",
    oms_type=OmsType.NETTING,
    account_type="MARGIN",
    base_currency="USD",
    starting_balances=["30_000 USD"],
)

# Configure a catalog for a live system
catalog_cfg = DataCatalogConfig(
    path=str(catalog.path),
    fs_protocol="file",
    name="local"
)

base_bar_type = BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL")
weekly_bar_type = BarType.from_str(f"{instrument_id}-1-WEEK-LAST-INTERNAL@1-MINUTE-EXTERNAL")
daily_bar_type = BarType.from_str(f"{instrument_id}-1-DAY-LAST-INTERNAL@1-MINUTE-EXTERNAL")

data = BacktestDataConfig(
    catalog_path=str(catalog.path),
    catalog_fs_protocol="file",
    data_cls=Bar,
    bar_types=[base_bar_type],
    instrument_id=instrument_id,
    start_time=start_ns,
    end_time=end_ns
)

engine = BacktestEngineConfig(
    strategies=[
        ImportableStrategyConfig(
            strategy_path="strategies.ict.ict_strategy:ICTStrategy",
            config_path="strategies.ict.ict_strategy:ICTStrategyConfig",
            config={
                "instrument_id": instrument_id,
                "base_bar_type": base_bar_type,
                "weekly_bar_type": weekly_bar_type,
                "daily_bar_type": daily_bar_type,
                "is_backtest": True,
            },
        ),
    ],
    logging=LoggingConfig(log_level="ERROR"),
    catalogs=[catalog_cfg]
)

config = BacktestRunConfig(
    engine=engine,
    venues=[venue],
    data=[data],
)

node = BacktestNode(configs=[config])

# run backtesting
elapsed_start = time.perf_counter()
# Runs one or many configs synchronously
results: list[BacktestResult] = node.run()
elapsed_end = time.perf_counter()

print(f"Elapsed time: {elapsed_end - elapsed_start:.6f} seconds")

Elapsed time: 1.972275 seconds


In [3]:
results

[BacktestResult(trader_id='BACKTESTER-001', machine_id='Yuriis-MacBook-Pro.local', run_config_id='e72c5346b0bc932f366dc2c1e680d59fe67c30d174565367dea5057d3c8108f1', instance_id='a1a8cbd0-59a5-4f05-b1c6-352a4b617a97', run_id='6c6507a7-1604-424d-8ba8-dccad54ef8ab', run_started=1765799123034635000, run_finished=1765799124865250000, backtest_start=1752019200000000000, backtest_end=1760918400000000000, elapsed_time=8899200.0, iterations=100467, total_events=0, total_orders=0, total_positions=0, stats_pnls={'USD': {'PnL (total)': 0.0, 'PnL% (total)': 0.0, 'Max Winner': 0.0, 'Avg Winner': 0.0, 'Min Winner': 0.0, 'Min Loser': 0.0, 'Avg Loser': 0.0, 'Max Loser': 0.0, 'Expectancy': 0.0, 'Win Rate': 0.0}}, stats_returns={'Returns Volatility (252 days)': nan, 'Average (Return)': nan, 'Average Loss (Return)': nan, 'Average Win (Return)': nan, 'Sharpe Ratio (252 days)': nan, 'Sortino Ratio (252 days)': nan, 'Profit Factor': nan, 'Risk Return Ratio': nan})]

In [4]:
backtest_engine: BacktestEngine = node.get_engine(config.id)
positions = backtest_engine.trader.generate_positions_report()

In [5]:
len(positions)

0

In [6]:
pd.set_option("display.max_rows", 200)   # show all rows
pd.set_option("display.max_columns", None)  # show all cols
positions

""


In [7]:
# Access portfolio analyzer
portfolio = backtest_engine.portfolio
fills_report = backtest_engine.trader.generate_fills_report()

# Get different categories of statistics
stats_pnls = portfolio.analyzer.get_performance_stats_pnls()
stats_returns = portfolio.analyzer.get_performance_stats_returns()
stats_general = portfolio.analyzer.get_performance_stats_general()

In [8]:
import matplotlib.pyplot as plt

positions_report = backtest_engine.trader.generate_positions_report()

if len(positions_report) > 0:
    # Plot cumulative returns
    returns = positions_report["realized_return"].cumsum()
    returns.plot(title="Cumulative Returns")
    plt.show()

    # Analyze fill quality (commission is a Money string e.g. "0.50 USD")
    # Extract numeric values and currency
    fills_report["commission_value"] = fills_report["commission"].str.split().str[0].astype(float)
    fills_report["commission_currency"] = fills_report["commission"].str.split().str[1]

    # Group by liquidity side and currency
    commission_by_side = fills_report.groupby(["liquidity_side", "commission_currency"])["commission_value"].sum()
    commission_by_side.plot.bar()
    plt.title("Commission by Liquidity Side and Currency")
    plt.show()
else:
    print("No positions to report - strategy has only context/filter rules, no entry signals generated.")

No positions to report - strategy has only context/filter rules, no entry signals generated.
